# AIR Homework 3 (Task 2)

## Setup

This notebook covers Task 2. Run the cells below before you start. They set up the notebook, load our working environment from `scene.py`, import your code, and recreate what we need from Task 1.

**NOTE:** The setup uses your code in `task1.py`, so complete Task 1 before you start this task.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

In [ ]:
# load our working environment from scene.py (see start_here.ipynb)
from scene import data, model

import mujoco
import mediapy as media
import numpy as np

In [ ]:
# import your code. this notebook calls your code from Task 1 through t1 (e.g.
# t1.MCTSNode), and your code for this task through t2 (e.g. t2.BlockStackingGymEnv)
import task1 as t1
import task2 as t2

In [ ]:
# from Task 1
import unittest
from copy import deepcopy

# used by the sanity checks
test = unittest.TestCase()


class CheckSimulator(t1.Simulator):
    """
    A fast stand-in for `BlockStackingEnv` without physics, used by the sanity checks.
    Each step stacks the next block at exactly the chosen offset. The episode ends when
    all four blocks are stacked, with a reward of 1 if all offsets are within 5 mm of the
    base block, and 0 otherwise.
    """

    def __init__(self):
        self.state = None

    def reset(self):
        self.state = t1.BlockStackingState(stacked_blocks_offsets=[np.zeros(2)])
        return deepcopy(self.state)

    def step(self, action):
        self.state = deepcopy(self.state)
        self.state.stacked_blocks_offsets.append(np.array([action.offset_x, action.offset_y]))
        done = self.done()
        success = done and all(np.abs(offset).max() <= 0.005 for offset in self.state.stacked_blocks_offsets)
        return deepcopy(self.state), float(success), done, {}

    def set_state(self, state):
        self.state = deepcopy(state)

    def sample_action(self, state):
        offset_x, offset_y = np.random.uniform(*t1.OFFSET_RANGE, size=2)
        return t1.BlockStackingAction(offset_x=offset_x, offset_y=offset_y)

    def done(self):
        return len(self.state.stacked_blocks_offsets) >= 4


# the block stacking environment, where we evaluate PUCT at the end of this task
env = t1.BlockStackingEnv(model, data)

## Task 2: RL-Guided MCTS

There are several issues with our DPW UCT implementation.
1. **Rollout Policy**: The zero-offset rollout policy is still quite naive. This means that our robot will prefer actions that are compatible with setting zero offset, which may deter it from exploring more balance-oriented actions.
2. **Action Sampling**: In both the expansion and selection stages, we sample actions uniformly at random. This means that we rely on pure chance to find good actions, which is inefficient in a large continuous action space.
3. **Simulation is Expensive**: Each simulation requires stepping the physics engine. While MuJoCo is known for its speed, it is still far more computationally expensive than most of our other computations. This limits the depth and breadth of our search tree.

To overcome these issues, we will implement a policy-guided version of UCT, called Predictor UCT (PUCT). The key idea is to learn a policy and value function that can guide the search process. The learned policy can be used to sample actions more intelligently during expansion and selection, while the learned value function can replace rollouts during simulation.

### Reinforcement Learning (RL) Agent

As a first step, we will implement a reinforcement learning (RL) agent to solve our problem. Specifically, we will use an actor-critic variant to learn both a policy (the actor) and a value function (the critic) simultaneously. For this, we will use the [Stable Baselines3](https://github.com/DLR-RM/stable-baselines3) library, which provides a convenient implementation of various RL algorithms with an easy-to-use interface compatible with [Gymnasium](https://gymnasium.farama.org/).

Stable Baselines3 is already installed in the course's conda environment, along with `tensorboard`, which we will use to visualize the training progress.

### Learning Environment

Stable Baselines3 is only compatible with Gymnasium environments, so we will need to adapt our block stacking environment accordingly.

`BlockStackingGymEnv` is provided in `task2.py`.

In [ ]:
gym_env = t2.BlockStackingGymEnv(model, data)
gym_env.reset()
media.show_image(gym_env.render())

### Cold Start

The RL agent will start training with no prior knowledge about the environment. This means that its initial policy will sample random actions. As we discussed regarding the UCT rollout policy, random actions may never lead to a successful tower. The RL agent cannot learn from failures alone, and must see some successful rollouts to understand what good actions look like.

To overcome this issue, we will implement a cold start mechanism. During the initial phase of training, the agent will only collect data using a heuristic-based policy. Even though the zero-offset policy performs poorly on its own, it can still produce some successful rollouts that the agent can learn from. After the this warmup phase, the agent will start collecting data using its own policy.

Stable Baselines3 provides a convenient way to intervene with every part of the training process using callbacks. We will implement a custom callback to handle the cold start mechanism.

`ZeroActionWarmupCallback` is provided in `task2.py`.

### Training

We are now ready to train our agent. Since we are using a cold start with a heuristic-based policy, what kind of RL algorithm should we use? If you were thinking of an off-policy algorithm, you are correct! Off-policy algorithms can learn from data collected by any policy, which is exactly what we need here. If we used an on-policy algorithm, the agent would not be able to learn from the heuristic-based rollouts since it expects data collected by its own policy. We will use the popular Soft Actor-Critic (SAC) algorithm for this task, which is an off-policy actor-critic method known for its stability and sample efficiency.

First, some housekeeping. Stable Baselines3 uses PyTorch as its backend, so we can leverage GPU acceleration if available. Beyond that, we need to be able to set a random seed for reproducibility.

`set_random_seed` and `device` are provided in `task2.py`.

In [ ]:
print(f"Using device: {t2.device}")

We are now ready to train our RL agent. The training loop is implemented for you below. Feel free to adjust the hyperparameters, though the ones provided should work well for this task. Remember, we are not trying to train a perfect agent. We just want a reasonably good policy and value function to guide our MCTS search, and as such, we want to train the agent with minimal effort and computational resources.

Using our chosen hyperparameters, training should take about 20-30 minutes. If you have already trained and saved the agent (see below), the cell below loads it from `rl_agent.zip` instead of retraining it.

In [ ]:
import os

from stable_baselines3 import SAC

if os.path.exists("rl_agent.zip"):
    print('loading existing RL agent from "rl_agent.zip"')
    rl_alg = SAC.load("rl_agent", device=t2.device)
else:
    print("training new RL agent")

    SEED = 42
    COLD_START_STEPS = 1_000
    NUM_TRAINING_STEPS = 3_000

    # set the random seed for reproducibility
    t2.set_random_seed(SEED)

    # reset the gym environment
    gym_env.reset()

    # initialize the RL algorithm
    rl_alg = SAC(
        "MlpPolicy",
        gym_env,
        verbose=1,                         # print training logs
        gamma=1.0,                         # no discounting for short episodic task
        learning_starts=COLD_START_STEPS,  # number of steps before training star
        ent_coef=0.01,                     # default entropy gets stuck on high value. set a lower fixed value.
        tensorboard_log="./tb/",           # log to tensorboard
        device=t2.device,                  # use selected device
        seed=SEED,                         # set random seed
    )

    # train the RL algorithm with zero-action warmup
    rl_alg.learn(
        total_timesteps=NUM_TRAINING_STEPS,
        callback=t2.ZeroActionWarmupCallback(warmup_steps=COLD_START_STEPS),
    )

We can view the training progress using TensorBoard. We can view the logs directly in this notebook by running the cell below. The output will be empty if you didn't train the agent in this session (e.g., if you loaded a pre-trained agent).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./tb

As you can see, the agent achieves much better performance than the zero-offset policy used during the cold start. With some more training and hyperparameter tuning, we can further improve its performance. However, while the agent is not perfect, it is good enough to guide our MCTS search effectively.

Let's try out our trained RL agent in the environment. We use the `predict` method to select actions according to the learned policy.

In [ ]:
obs, _, = gym_env.reset()

done = False
frames = []
while not done:
    action, _ = rl_alg.predict(obs, deterministic=True)
    print(f"Taking action: {action}")
    obs, reward, done, _, info = gym_env.step(action, framerate=10)
    print(f"Reward: {reward}, Done: {done}")
    frames += info['frames']

media.show_video(frames, fps=10)

Let's save our trained agent to avoid repeating the training process.

In [ ]:
rl_alg.save('rl_agent')

### PUCT Implementation

PUCT is similar to UCT, with a few key differences:
1. **Action Sampling**: During expansion and selection, instead of sampling actions uniformly at random, we sample actions according to the learned policy. Specifically, we sample actions from a Gaussian distribution centered at the action mean predicted by the policy, with a fixed standard deviation.
2. **Simulation**: Instead of performing rollouts during simulation, we use the learned value function to estimate the value of the selected node.
3. **UCB Calculation**: The UCB formula is modified to incorporate the prior probability of each action according to the learned policy. First, if the edge has no visits, we don't set its UCB to infinity as we did before, but instead take its prior probability into account. Second we modify UCB formula to prefer exploring actions with high prior probability deeper in the tree. The modified PUCT formula is as follows:
$$
PUCB(s, a) = Q(s, a) + C_{\text{puct}} \cdot P(s, a) \sqrt{\frac{\ln N_{s}}{n_{s,a}}}
$$

Implement a new edge type for PUCT that calculates UCB values using the PUCT formula described above:

**TODO:** implement `MCTSEdgeWithPrior` in `task2.py`.

In [ ]:
# sanity check: MCTSEdgeWithPrior
root = t1.MCTSNode(state=None, visits=10)
edge = t2.MCTSEdgeWithPrior(parent=root, action=None, outcomes=[], visits=2, total_return=1.0, prior=0.5)

# the prior scales the exploration term
test.assertAlmostEqual(edge.ucb(2 ** 0.5), 0.5 + 0.5 * 2 ** 0.5 * np.sqrt(np.log(10) / 2))
test.assertAlmostEqual(edge.ucb(0.0), 0.5)
edge.prior = 2.0
test.assertAlmostEqual(edge.ucb(2 ** 0.5), 0.5 + 2.0 * 2 ** 0.5 * np.sqrt(np.log(10) / 2))

# an edge that was never visited gets its prior
unvisited_edge = t2.MCTSEdgeWithPrior(parent=root, action=None, outcomes=[], prior=0.3)
test.assertAlmostEqual(unvisited_edge.ucb(2 ** 0.5), 0.3)

print(f"MCTSEdgeWithPrior sanity checks passed. PUCB value = {edge.ucb(2 ** 0.5):.3f}")

To implement the changes to UCT, we need a way to get the action distribution parameters and value estimate from the RL agent for a given state. While SAC provides an easy way to get the prior distribution, it does not provide a direct way to get the value estimate for a state because it learns a Q-function instead of a V-function. To calcualte the Q-function, we would need to enumerate all possible actions, which is not possible in a continuous action space. Therefore, we will use the Q-value of the most probable action as a proxy for the value estimate. Remember, its not about getting perfect estimates, but rather about guiding the search effectively.

**NOTE**: SAC uses a technique called "double Q-learning", where two separate Q-functions are learned to mitigate overestimation bias. Therefore, when querying the Q-value, we will take the minimum of the two Q-functions to avoid overestimating the state value.

`get_action_dist` and `get_state_value` are provided in `task2.py`.

We will now make changes to our selection and expansion methods to use the learned policy for sampling. Instead of reimplementing all methods, we will update our `MCTSNode` objects with the value of the state represented by that node, and the action distribution parameters (mean and standard deviation) that state. We will do this immediately when a node is created.

Implement the necessary changes to selection and expansion to use the learned policy for action sampling and the learned value function for simulation:

**TODO:** implement `dpw_select_outcome_and_set_priors_and_values`, `dpw_select_node_with_action_priors`, and `dpw_expand_node_with_action_prior` in `task2.py`. `set_node_guiding_values` is provided there.

In [ ]:
# sanity check: dpw_select_outcome_and_set_priors_and_values, dpw_select_node_with_action_priors,
# and dpw_expand_node_with_action_prior
check_sim = CheckSimulator()


def has_guiding_values(node):
    return getattr(node, "value", None) is not None and getattr(node, "actions_dist", None) is not None


# a root that was never visited is selected, with its guiding values set
root = t1.MCTSNode(state=check_sim.reset())
selected = t2.dpw_select_node_with_action_priors(root, check_sim, 2 ** 0.5, 1.0, 0.5, rl_alg)
test.assertTrue(selected is root, "selection should stop at a root that was never visited")
test.assertTrue(has_guiding_values(root), "the selected root has no guiding values")

# expanding adds an edge with a prior, and one outcome with guiding values
root = t1.MCTSNode(state=check_sim.reset())
t2.set_node_guiding_values(root, check_sim, rl_alg)
outcome = t2.dpw_expand_node_with_action_prior(root, check_sim, rl_alg)
test.assertEqual(len(root.outgoing_edges), 1)
edge = root.outgoing_edges[0]
test.assertTrue(np.isfinite(edge.prior) and edge.prior > 0, "the new edge has no valid prior")
test.assertTrue(len(edge.outcomes) == 1 and edge.outcomes[0] is outcome, "the new outcome is not in edge.outcomes")
test.assertTrue(outcome.incoming_edge is edge, "the incoming edge of the new outcome is wrong")
np.testing.assert_allclose(outcome.state.stacked_blocks_offsets[-1], [edge.action.offset_x, edge.action.offset_y])
test.assertTrue(has_guiding_values(outcome), "the expanded outcome has no guiding values")
expanded_prior = edge.prior

# with four visits, the PW limit of the edge is 2, so a new outcome is expanded and gets
# guiding values
edge.visits, outcome.visits = 4, 4
new_outcome = t2.dpw_select_outcome_and_set_priors_and_values(edge, check_sim, 1.0, 0.5, rl_alg)
test.assertEqual(len(edge.outcomes), 2, "with four visits, a new outcome should be expanded")
test.assertTrue(edge.outcomes[1] is new_outcome, "the new outcome should be selected")
test.assertTrue(has_guiding_values(new_outcome), "the new outcome has no guiding values")

# selection continues from a root with one visit and one edge to the edge's only outcome,
# and sets its guiding values
root = t1.MCTSNode(state=check_sim.reset(), visits=1)
t2.set_node_guiding_values(root, check_sim, rl_alg)
edge = t2.MCTSEdgeWithPrior(
    parent=root, action=t1.BlockStackingAction(offset_x=0.0, offset_y=0.0), outcomes=[], visits=1, prior=1.0
)
root.outgoing_edges.append(edge)
check_sim.set_state(root.state)
child = t1.MCTSNode(state=check_sim.step(edge.action)[0], incoming_edge=edge, visits=1)
edge.outcomes.append(child)
selected = t2.dpw_select_node_with_action_priors(root, check_sim, 2 ** 0.5, 1.0, 0.5, rl_alg)
test.assertTrue(selected is child, "selection should continue from the root to its only outcome")
test.assertTrue(has_guiding_values(child), "the selected outcome has no guiding values")

print(
    "PUCT selection and expansion sanity checks passed. "
    f"value of the root = {float(root.value):.3f}, prior of the expanded action = {expanded_prior:.3f}"
)

We can now implement the PUCT pipeline. This is implemented for you below. Note the differences from the UCT implementation: selection and expansion are done using the learned policy while also recording the values of the nodes, and simulation is done using the learned value function instead of rollouts.

`ContinuousActionStochasticOutcomePUCT` is provided in `task2.py`.

Let's see how well PUCT performs in the environment. We use the same evaluation loop as before, but replace UCT with PUCT. Can PUCT perform better with less planning time?

**TODO:** set `PLANNING_TIME` in `task2.py`.

In [ ]:
# set PLANNING_TIME in task2.py

env.reset()
media.show_image(env.render())

sim_env = env.clone()

puct = t2.ContinuousActionStochasticOutcomePUCT(
    sim=sim_env,
    rl_alg=rl_alg,
    pw_k=1.0,
    pw_alpha=0.5,
)

root_node = t1.MCTSNode(state=sim_env.state)
nodes, frames = t1.run_mcts_planning(
    env,
    mcts=puct,
    root_node=root_node,
    planning_time=t2.PLANNING_TIME,
    exploration_constant=2 ** 0.5,  # common choice for exploration constant in PUCT
    framerate=0
)

## Conclusion

In this notebook, we implemented the UCT algorithm with Double Progressive Widening to handle continuous action spaces. We then enhanced our MCTS implementation by integrating a reinforcement learning agent to guide the search process, resulting in the Predictor UCT (PUCT) algorithm. Our PUCT implementation is actually similar to the approach used in AlphaZero, but training an RL agent separately rather than training the policy and value function jointly with MCTS. As we saw, PUCT can achieve better performance with less planning time compared to UCT, thanks to the guidance provided by the learned policy and value function. This demonstrates the power of combining planning and learning in solving complex decision-making problems.